In [1]:
import torch

tensor1 = torch.tensor([1., 2., 3.])
print('tensor1 자동미분 사용 여부: ', tensor1.requires_grad)

tensor2 = torch.tensor([1., 2., 3.], requires_grad=False)
print('tensor2 자동미분 사용 여부: ', tensor2.requires_grad)

tensor3 = torch.tensor([1., 2., 3.], requires_grad=True)
print('tensor3 자동미분 사용 여부: ', tensor3.requires_grad)

tensor1 자동미분 사용 여부:  False
tensor2 자동미분 사용 여부:  False
tensor3 자동미분 사용 여부:  True


# torch Tensor의 requires_grad란?
requires_grad는 이 텐서에 대해 미분(기울기) 계산을 추적할지 결정하는 스위치입니다.

## 예시 파일
[PyTorch MNIST 예시 파일](https://raw.githubusercontent.com/pytorch/examples/main/mnist/main.py)

## 답변
핵심만 보면 다음과 같습니다.

1. requires_grad=True
- 연산 그래프를 만들어 역전파 시 기울기를 계산합니다.
- 보통 학습할 파라미터(가중치, 편향)에 설정합니다.
- loss.backward() 후 해당 리프 텐서의 grad에 값이 쌓입니다.

2. requires_grad=False
- 기울기 추적을 하지 않습니다.
- 고정된 상수, 전처리용 텐서, 추론 단계에서 주로 사용합니다.
- 메모리와 연산량을 줄일 수 있습니다.

짧은 예시:
- w = torch.tensor(2.0, requires_grad=True)
- x = torch.tensor(3.0)
- y = w * x
- y.backward()
- 결과: w.grad = 3.0

실무에서 자주 같이 쓰는 것:
1. torch.no_grad()
- 블록 내부 연산 전체를 미분 추적 없이 수행(평가/추론용).

2. detach()
- 현재 그래프에서 텐서를 분리해 이후 연산 추적을 끊음.

3. model.eval()과의 차이
- eval은 드롭아웃/배치정규화 동작 모드 전환.
- requires_grad/no_grad는 미분 추적 on/off.
- 서로 역할이 다릅니다.

### 추가 자료
- [PyTorch Autograd 공식 튜토리얼](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)
- [torch.Tensor.requires_grad 공식 문서](https://pytorch.org/docs/stable/generated/torch.Tensor.requires_grad.html)

In [2]:
# 미분 가능한 텐서 x를 생성
x = torch.tensor(5.0, requires_grad=True)

# 함수 f 정의: f(x) = x^2
f = x ** 2

# 함수 f에 대한 x의 미분 계산
f.backward()

# x에 대한 f의 미분값 출력
print(f"x = {x}에서의 미분 값: {x.grad}")

x = 5.0에서의 미분 값: 10.0


# 손실함수의 미분의 특징 

- 기울기가 음수 -> 매개 변수에 대해 손실이 감소하는 방향
- 모델의 매개 변수를 기울기가 가리키는 방향으로 조정하면, 손실 함수

In [4]:
import torch
import torch.nn.functional as F

# 모델의 예측값 파라미터 정의
prediction1 = torch.tensor([3.0], requires_grad=True)
prediction2 = torch.tensor([10.0], requires_grad=True)

# 실제 값 레이블 정의
actual_value = torch.tensor([2.0])

# 첫 번째 예측값에 대한 MSE 손실 계산
mse_loss1 = F.mse_loss(prediction1, actual_value)

# 두 번째 예측값에 대한 MSE 손실 계산
mse_loss2 = F.mse_loss(prediction2, actual_value)

# 첫 번째 손실에 대한 미분 계산
mse_loss1.backward() # 손실 함수에 대한 back propagation을 수행 

# 두 번째 손실에 대한 미분 계산
mse_loss2.backward() # 손실 함수에 대한 back propagation을 수행 

# 미분값(그래디언트) 출력
gradient_pred_1 = prediction1.grad.item()
gradient_pred_2 = prediction2.grad.item()

print(f'첫 번째 예측값의 MSE 손실 함수 미분값: {gradient_pred_1}')
print(f'두 번째 예측값의 MSE 손실 함수 미분값: {gradient_pred_2}')

첫 번째 예측값의 MSE 손실 함수 미분값: 2.0
두 번째 예측값의 MSE 손실 함수 미분값: 16.0


# F.mse_loss

$ \text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y_i})^2 $

## 미분값이 2.0 인 경우 

- 파라미터가 실제 레이블과 비교하여 비교적 작은 오차를 가지고 있으며, 손실 함수를 최소화 하기 위해 파라미터를 업데이트 할 때 상대적으로 작은 조정이 필요함을 나타냄. 
- 모델이 비교적 좋은 예측을 하고 있으며 작은 조정으로 충분할 수 있다. 

## 미분값이 16.0 인 경우

- 파라미터가 실제 레이블과 상당히 큰 오차를 가지고 있으며, 손실 함수를 최소화 하기 위해 파라미터를 업데이트할 때 더 큰 조정이 필요함을 나나냄. 

# Categorical Cross-Entropy, CCE

In [9]:
import torch
import torch.nn as nn

# 3개 클래스에 대한 모델 출력값(로짓)입니다.
# 로짓은 softmax를 거치기 전의 점수이며, requires_grad=True로 두면 역전파 때 미분이 계산됩니다.
logits = torch.tensor([0.1, 0.2, 0.7], requires_grad=True)

# 정답 클래스의 인덱스입니다.
# CrossEntropyLoss는 one-hot 벡터가 아니라 클래스 번호를 받습니다.
target = torch.tensor([0])

# 다중 분류에서 자주 쓰는 손실 함수입니다.
cross_entropy_loss = nn.CrossEntropyLoss()

# CrossEntropyLoss는 입력을 보통 [배치 크기, 클래스 수] 형태로 기대합니다.
# 현재 logits는 [클래스 수] 형태이므로, unsqueeze(0)으로 앞에 배치 차원 1개를 추가합니다.
loss = cross_entropy_loss(logits.unsqueeze(0), target)

# 손실값에 대해 역전파를 수행해 logits의 기울기를 계산합니다.
loss.backward()

# 각 클래스 로짓에 대한 기울기 확인
gradient_logits = logits.grad

print(f'로짓 세트의 CCE 손실 함수 미분값: {gradient_logits}')

로짓 세트의 CCE 손실 함수 미분값: tensor([-0.7454,  0.2814,  0.4640])


# 왜 `unsqueeze(0)`이 필요한가?

`CrossEntropyLoss`는 보통 입력을 `배치 크기 × 클래스 수` 형태로 받습니다. 즉, 모델이 한 번에 여러 샘플을 처리한다고 가정합니다.

여기서 `logits = [0.1, 0.2, 0.7]`은 **샘플 1개**의 클래스 점수입니다. 하지만 `CrossEntropyLoss` 입장에서는 이 값이 그냥 `클래스 수`만 있는 1차원 벡터처럼 보이기 때문에, 보통은 `[[0.1, 0.2, 0.7]]`처럼 **배치 차원 1개**를 더해 주는 편이 안전합니다.

`unsqueeze(0)`는 맨 앞에 크기 1인 차원을 추가합니다.

- 원래: `(3,)`
- `unsqueeze(0)` 후: `(1, 3)`

즉, "샘플 1개를 가진 배치"로 바꿔 주는 역할입니다.

## 배치 처리란?

배치 처리(batch processing)는 데이터를 **한 개씩** 처리하지 않고, **여러 개를 묶어서 한 번에** 처리하는 방식입니다.

예를 들어:

- 샘플 1개씩 학습: 느림, 비효율적일 수 있음
- 샘플 32개씩 묶어서 학습: 더 빠르고 GPU 활용에 유리함

그래서 딥러닝에서는 보통 입력의 첫 번째 차원을 배치 차원으로 둡니다.

예시:

- `x.shape = (32, 784)` → 32개의 이미지가 한 번에 들어옴
- `logits.shape = (32, 10)` → 각 이미지마다 10개 클래스 점수 출력
- `target.shape = (32,)` → 각 샘플의 정답 클래스 번호

## 왜 지금 코드에서는 배치 차원이 꼭 보이게 해야 하나?

`CrossEntropyLoss`는 내부적으로 각 샘플에 대해 손실을 계산한 뒤 평균을 내는 구조입니다. 그래서 입력과 정답의 차원이 맞아야 합니다.

이 코드에서는 샘플이 1개뿐이므로 `unsqueeze(0)`으로 배치 차원을 만들어서

- 입력: `(1, 3)`
- 정답: `(1,)`

형태로 맞춰 주는 것입니다.

## 아주 짧은 예시

- 클래스 3개일 때 정답이 0번 클래스라면
- logits가 `[0.1, 0.2, 0.7]`일 때
- `unsqueeze(0)`을 통해 `[[0.1, 0.2, 0.7]]`로 바꿔서 손실을 계산합니다.

이렇게 하면 "샘플 1개짜리 미니배치"로 처리할 수 있습니다.

In [8]:
import torch

# 1차원 텐서: 원래 shape는 (3,)
logits = torch.tensor([0.1, 0.2, 0.7])
print("원래 shape:", logits.shape)

# 맨 앞에 배치 차원 1개 추가: shape가 (1, 3)으로 바뀜
logits_batch = logits.unsqueeze(0)
print("unsqueeze(0) 후 shape:", logits_batch.shape)
print("unsqueeze(0) 후 값:\n", logits_batch)

원래 shape: torch.Size([3])
unsqueeze(0) 후 shape: torch.Size([1, 3])
unsqueeze(0) 후 값:
 tensor([[0.1000, 0.2000, 0.7000]])


[0.1, 0.2, 0.7] → shape는 (3,)  
[[0.1, 0.2, 0.7]] → shape는 (1, 3)